# Re-run: fairness and stability (RQ5)

From the benchmark repository's committed outputs: `fairness-ellipse/out/result.json` (D, the model's gap
minus rater 2's gap at the same rater-1 grade, per contrast and model), `fairness-ellipse/out/rep-result.json`
(the replication), `disability/out/result.json` and `out/public/drift.json`.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grading-benchmark-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


SOURCE_ENV, SOURCE_NAME = "GBENCH_DIR", "ai-grading-benchmark"   # set when a release's code is another repository


def source_repo():
    """The repository a release's code and committed outputs are published in (a release whose paper cites
    it rather than a copied code package): $SOURCE_ENV, or a checkout named SOURCE_NAME beside this
    notebook or beside a folder above it."""
    here = Path.cwd()
    places = [Path(os.environ[SOURCE_ENV])] if SOURCE_ENV and os.environ.get(SOURCE_ENV) else []
    places += [p / SOURCE_NAME for p in [here, *here.parents]]
    for d in places:
        if d.is_dir() and (d / "out").is_dir():
            return d
    raise FileNotFoundError(
        f"Could not find the {SOURCE_NAME} repository this release's code is published in. Clone it at the "
        f"commit the release file names, then set the environment variable {SOURCE_ENV} to the clone, or "
        f"run this notebook from a folder beside it.")


def load_source_json(relpath):
    """A committed output of that repository, by its path there (cited in the paper as code/<path>)."""
    return json.loads((source_repo() / relpath).read_text(encoding="utf-8"))


print("Source repository:", shown(source_repo()))

Source repository: ../../../../../../ai-grading-benchmark


In [2]:
E = load_source_json("fairness-ellipse/out/result.json")["contrasts"]
c1 = E["C1"]["graders"]
assert len(c1) == 22 and all(g["verdict"] == "NO MORE THAN A SECOND HUMAN" for g in c1.values())
c3 = E["C3"]["graders"]
below = sum(g["D"]["ci95"][1] < 0 for g in c3.values())
wholly = sum(g["D"]["ci95"][1] < -0.10 for g in c3.values())
print("C3 Hispanic against White: interval below 0 for", below, "of 22; wholly below -0.10 for", wholly)
assert (below, wholly) == (18, 0)
rep = load_source_json("fairness-ellipse/out/rep-result.json")
print("replication verdict:", rep["verdict"])
assert "MIXED" in str(rep["verdict"])

C3 Hispanic against White: interval below 0 for 18 of 22; wholly below -0.10 for 0
replication verdict: MIXED


In [3]:
dg = load_source_json("disability/out/result.json")
assert len(dg) == 22 and all(v["ci95"][1] < 0 for v in dg.values())
rep_n = sum(v["verdict"] == "GAP REPLICATES" for v in dg.values())
print("disability: every model grades the identified group lower; GAP REPLICATES for", rep_n)
assert rep_n == 20
mc = load_source_json("disability-mechanism/out/mechanism.json")["models"]
part = [v for v in mc.values() if v["verdict_B"] == "SURFACE FEATURES ACCOUNT FOR PART OF THE GAP"]
fifth = sum(v["share"]["estimate"] <= 0.2 for v in part)
print("surface features carry part of the gap for", len(part), "models; at most a fifth for", fifth,
      "; reverse regression:", {v["verdict_A"] for v in mc.values()})
assert (len(part), fifth) == (20, 17) and {v["verdict_A"] for v in mc.values()} == {"UNMEASURED"}
assert all(v["gap_B1"]["ci95"][1] < 0 for v in part)

disability: every model grades the identified group lower; GAP REPLICATES for 20
surface features carry part of the gap for 20 models; at most a fifth for 17 ; reverse regression: {'UNMEASURED'}


In [4]:
dr = load_source_json("out/public/drift.json")
shift = sum(d["lo"] > 0 or d["hi"] < 0 for d in dr)
print(len(dr), "comparisons;", shift, "shift the mean; across below within in all:", all(d["qwk_across"] < d["qwk_within"] for d in dr))
assert (len(dr), shift) == (7, 4) and all(d["qwk_across"] < d["qwk_within"] for d in dr)
assert {d["shift"] > 0 for d in dr if d["lo"] > 0 or d["hi"] < 0} == {True, False}   # both directions

7 comparisons; 4 shift the mean; across below within in all: True
